# nb_mdf_05_get_worklist - what should the pipeline process?

Called by the **pl_mdf_orchestrator** pipeline. Reads the control tables and returns (via `notebookutils.notebook.exit`)
a JSON object: `items` (the array the pipeline's ForEach iterates over) and `object_ids`
(passed to a single Silver run so reference data is processed before the transactions that depend on it). All decision logic stays in metadata + code under source
control; the pipeline only orchestrates.

For `copy_db` objects it also builds the **incremental source query** from the stored source watermark, e.g.
`SELECT * FROM src.assets WHERE last_modified > '2026-11-03 16:00:00'`.

`rerun_failed_only = True` returns only objects whose most recent run failed - the operational "rerun what broke" button.

In [ ]:
load_group = "all"
object_ids = ""
rerun_failed_only = False
run_date = ""            # YYYY-MM-DD for the landing folder; empty = today (Australia/Sydney)

In [ ]:
%run nb_mdf_utils

In [ ]:
from zoneinfo import ZoneInfo
run_date = run_date or datetime.now(ZoneInfo("Australia/Sydney")).strftime("%Y-%m-%d")
objects = get_objects(object_ids, load_group)

if rerun_failed_only:
    w = Window.partitionBy("object_id").orderBy(F.col("started_at").desc())
    latest = (spark.table(tname("meta", "run_log")).where("layer IN ('bronze','silver')")
              .withColumn("rn", F.row_number().over(w)).where("rn = 1 AND status = 'FAILED'"))
    failed_ids = {r["object_id"] for r in latest.select("object_id").collect()}
    objects = [o for o in objects if o["object_id"] in failed_ids]

work = []
for o in objects:
    item = {"object_id": o["object_id"], "ingest_method": o["ingest_method"], "load_group": o["load_group"],
            "priority": o["priority"], "source_query": "", "landing_folder": "", "landing_file": ""}
    if o["ingest_method"] == "copy_db":
        wm = get_watermark(o["object_id"], "source") if o["load_type"] == "incremental" else None
        q = f"SELECT * FROM {o['source_table']}"
        if wm and o.get("watermark_column"):
            q += f" WHERE {o['watermark_column']} > '{wm}'"
        item["source_query"] = q
        item["landing_folder"] = f"{o['landing_path']}/load_date={run_date}"
        item["landing_file"] = f"{o['source_object']}_{run_date.replace('-', '')}.csv"
    work.append(item)

# items -> ForEach (copy + bronze per object, in parallel); object_ids -> one Silver call that runs in priority order
result = {"items": work, "object_ids": ",".join(w["object_id"] for w in work), "count": len(work)}
print(json.dumps(result, indent=1))
notebookutils.notebook.exit(json.dumps(result))